# Delivery Performance Validation

**Owner:** Shreyansh Pankaj  
**Assigned reviewer:** Hazim Ali  
**Run after:** `03_gold_eda.ipynb`

Validates delivery fields, distance-band coverage, sort order, rate bounds, and the executive on-time KPI.

This notebook is an owner-specific PySpark contribution. The owner must run it personally, inspect the displayed result, understand every assertion, and commit it from their own GitHub account.


## 1. Load the validated project tables


In [ ]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "analytics"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

deliveries = spark.table(f"{CATALOG}.{SCHEMA}.deliveries_silver")
orders_gold = spark.table(f"{CATALOG}.{SCHEMA}.orders_gold")
delivery_distance = spark.table(f"{CATALOG}.{SCHEMA}.delivery_distance_gold")
kpi = spark.table(f"{CATALOG}.{SCHEMA}.kpi_gold")

print(f"deliveries_silver: {deliveries.count():,}")
print(f"orders_gold: {orders_gold.count():,}")
print(f"delivery_distance_gold: {delivery_distance.count():,}")


## 2. Run owner-specific reconciliation and integrity checks


In [ ]:
# Clean delivery records must have valid distance/promise values and non-negative delay.
assert deliveries.filter(F.col("DistanceKm") <= 0).count() == 0
assert deliveries.filter(F.col("PromisedMinutes") <= 0).count() == 0
assert deliveries.filter(F.col("DelayMinutes") < 0).count() == 0

# Every delivered record with an actual time must have a Boolean on-time result.
delivered_with_actual = deliveries.filter(
    (F.col("DeliveryStatus") == "Delivered") & F.col("ActualMinutes").isNotNull()
)
assert delivered_with_actual.filter(F.col("IsOnTime").isNull()).count() == 0

# Distance bands must cover all Gold orders that have an actual delivery time.
gold_with_actual = orders_gold.filter(F.col("ActualMinutes").isNotNull())
band_delivery_total = delivery_distance.agg(F.sum("Deliveries")).first()[0]
assert band_delivery_total == gold_with_actual.count()

sort_orders = [row["SortOrder"] for row in delivery_distance.orderBy("SortOrder").select("SortOrder").collect()]
assert sort_orders == list(range(len(sort_orders)))
assert delivery_distance.filter(
    ~F.col("OnTimeRatePct").between(0, 100)
).count() == 0
assert delivery_distance.filter(F.col("AverageDelayMinutes") < 0).count() == 0

# Independently recalculate overall on-time rate and compare with kpi_gold.
recomputed_on_time = gold_with_actual.agg(
    F.round(F.avg(F.col("IsOnTime").cast("double")) * 100, 2).alias("Rate")
).first()["Rate"]
published_on_time = kpi.first()["OnTimeDeliveryRatePct"]
assert abs(recomputed_on_time - published_on_time) < 0.01


## 3. Display the observed business result and success marker


In [ ]:
print(f"Delivered Gold orders validated: {gold_with_actual.count():,}")
print(f"Published on-time rate: {published_on_time:.2f}%")
print(f"Recomputed on-time rate: {recomputed_on_time:.2f}%")
display(
    delivery_distance.orderBy("SortOrder").select(
        "DistanceBand", "Deliveries", "OnTimeRatePct", "AverageDelayMinutes", "SortOrder"
    )
)

print("SHREYANSH_DELIVERY_VALIDATION_PASSED")


## What the owner must be able to explain

- Which tables were compared and why.
- What each assertion protects against.
- What the displayed result means for FreshRoute.
- Why the final success marker `SHREYANSH_DELIVERY_VALIDATION_PASSED` only prints after every check passes.
